# 🎬 Discipline Film Factory — 5 production videos on Kaggle 2×T4, powered by Groq

**What this notebook does:** it takes five complete film blueprints (scripts, scene plans, palettes,
music scores — all deterministic, all committed to the repo), hands them to a **Groq-powered AI
director** (script polish → scene art direction → score mood → publish metadata), then produces
**five 2-minute, production-ready videos about discipline** through the OpenVideoHarness pipeline:

> Groq director → edge-tts narration (free) → adaptive timing to exactly 120.0 s →
> code-composed score + SFX → 3-bus mix at −14 LUFS → 720p draft + Groq **vision QA** on the
> contact sheet → **final 1080p render on both T4s** (NVENC p7/cq19, one card per half,
> determinism-gated) → mux → `bin/vh check` → YouTube delivery copy + metadata.

**Requirements (set before running):**
- **Accelerator: GPU T4 x2** (Settings → Accelerator)
- **Internet: ON** (Settings → Internet) — needed for edge-tts, Groq, git, npm packages
- Runtime: ~60–100 min for all five films (boot ~5–10 min on a cold cache, seconds after)

**No API key?** The factory ships the blueprints and degrades gracefully: with no Groq the films
still get made exactly as designed (the AI passes are pure upside).

> ⚠️ The Groq key below makes this notebook work out of the box in a **private** kernel.
> If you ever make the kernel public, attach the key as a Kaggle secret instead
> (Add-ons → Secrets → `GROQ_API_KEY`) and clear the inline default.

## 1 · Configuration — everything you'd want to change lives here

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CONFIG — one cell to rule the whole run
# ══════════════════════════════════════════════════════════════════════════

GROQ_API_KEY = ""   # ← tries Kaggle Secrets first, then this inline default

# ── the Groq models: "auto" picks the first available from the preference
#    lists below (checked against the live /models at boot). To pin a model,
#    write its id here; to try a brand-new one, put it at the FRONT of a list.
GROQ_TEXT_MODEL = "auto"
GROQ_VISION_MODEL = "auto"
TEXT_MODEL_PREFERENCE = [          # fastest-first, checked top-down
    "openai/gpt-oss-120b",
    "llama-3.3-70b-versatile",
    "openai/gpt-oss-20b",
    "qwen/qwen3-32b",
    "llama-3.1-8b-instant",
]
VISION_MODEL_PREFERENCE = [        # Groq's vision models (frame QA)
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "meta-llama/llama-4-maverick-17b-128e-instruct",
]

# ── the films
FILMS_TO_MAKE = "all"              # "all" | a list of slugs, e.g. ["the-1-percent-rule"]
TARGET_SECONDS = 120.0             # the master lands here (adaptive gap retry enforces it)
FINAL_TIER = "final"               # "final" = 1080p NVENC p7/cq19 master | "beast" = 720p ship grade
DRAFT_VISION_QA = True             # 720p draft + contact sheet + Groq vision rubric before the master
AI_DIRECTOR = True                 # the four Groq passes (script/scene/score/metadata)

# ── where things live on Kaggle
import os
BASE = "/kaggle/working"     # everything persists here between sessions
OVH = f"{BASE}/OpenVideoHarness"          # the one clone: workbench + factory + blueprints
FACTORY_REPO = OVH                        # the factory package lives at {OVH}/factory
OUT_DIR = f"{BASE}/discipline-films"
os.makedirs(OUT_DIR, exist_ok=True)

# Kaggle secret (if attached) wins over the inline key
try:
    from kaggle_secrets import UserSecretsClient
    GROQ_API_KEY = UserSecretsClient().get_secret("GROQ_API_KEY") or GROQ_API_KEY
    print("Groq key loaded from Kaggle Secrets")
except Exception:
    pass

print(f"key: {'✓ present' if GROQ_API_KEY else '✗ absent (offline mode: blueprints ship as-is)'}")
print(f"target: {TARGET_SECONDS}s · final tier: {FINAL_TIER} · vision QA: {DRAFT_VISION_QA} · director: {AI_DIRECTOR}")

## 2 · Boot the session — the repo + the render stack

`tools/kaggle/session.sh` is OpenVideoHarness's one-command boot: it checks the GPUs, installs
Node 22 + a Chrome headless build (no sudo, everything into persistent caches), probes **NVENC per
card**, and writes `.kaggle/env.sh`. The film factory (`factory/` in the same repo) comes
with the clone. We skip the local VLM server — vision QA runs on Groq's vision models instead,
so GPU 1 stays free for NVENC + renders.

In [ ]:
import subprocess, os

def run(cmd, cwd=None, timeout=60 * 25):
    print(f"$ {cmd}")
    p = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True, timeout=timeout)
    out = (p.stdout or "") + (p.stderr or "")
    print(out[-3000:] if len(out) > 3000 else out)
    if p.returncode != 0:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")
    return p.stdout

os.makedirs(BASE, exist_ok=True)
os.chdir(BASE)

# clone (or fast-forward) the workbench + factory — public, no tokens needed
if not os.path.isdir(OVH):
    run(f"git clone https://github.com/coreselfmastery/OpenVideoHarness.git {OVH}")
else:
    run(f"git -C {OVH} pull --ff-only || true", timeout=120)

# the machine glance + render stack boot (idempotent; caches persist in /kaggle/working)
run(f'bash "{OVH}/tools/kaggle/session.sh" --no-vlm', cwd=OVH, timeout=60 * 30)
run("nvidia-smi -L || true")
print(open(f"{OVH}/.kaggle/env.sh").read())

## 3 · Connect the AI — Groq model check + auto-selection

This tests the key end-to-end with a real completion, lists what the account can see, and resolves
`auto` against the preference lists. If Groq is unreachable, the run continues in offline mode —
the deterministic blueprints carry the films.

In [ ]:
import sys, json
sys.path.insert(0, FACTORY_REPO)

from factory.groq_client import GroqClient
from factory import groq_client as gc

groq = None
groq_client_models = []
if GROQ_API_KEY:
    try:
        groq = GroqClient(GROQ_API_KEY)
        ping = groq.ping()
        print(f"✓ Groq alive · {ping['latency_s']}s round-trip")
        groq_client_models = groq.list_models()
        print(f"✓ {len(groq_client_models)} models visible on this key")
        resolved = groq.resolve_models(TEXT_MODEL_PREFERENCE, VISION_MODEL_PREFERENCE)        # honour an explicit pin over auto
        if GROQ_TEXT_MODEL != "auto":
            groq.text_model = GROQ_TEXT_MODEL
        if GROQ_VISION_MODEL != "auto":
            groq.vision_model = GROQ_VISION_MODEL
        print(f"✓ text model   → {groq.text_model}   (change GROQ_TEXT_MODEL / TEXT_MODEL_PREFERENCE above)")
        print(f"✓ vision model → {groq.vision_model} (change GROQ_VISION_MODEL / VISION_MODEL_PREFERENCE above)")
        if not groq.vision_model:
            print("! no vision model available — frame QA will be skipped (non-fatal)")
    except Exception as e:
        print(f"✗ Groq unreachable ({str(e)[:200]}) — running in OFFLINE mode")
        groq = None
else:
    print("no key — OFFLINE mode: the blueprints ship exactly as designed")

for m in sorted(groq_client_models)[:40] if groq else []:
    tag = " ← text" if m == groq.text_model else (" ← vision" if m == groq.vision_model else "")
    print("  ", m, tag)

## 4 · The five films

Each blueprint is a complete, deterministic film: narration script (word-budgeted for a 120 s
master), one scene plan per line (8 scene kinds), a palette, a code-composed score, and publish
metadata. The Groq director enriches — never replaces — this.

In [ ]:
from factory import FILMS

selected = FILMS if FILMS_TO_MAKE == "all" else [f for f in FILMS if f["slug"] in FILMS_TO_MAKE]
assert selected, f"no films matched {FILMS_TO_MAKE}"

for i, f in enumerate(selected, 1):
    words = sum(len(t.split()) for _, t in f["script"])
    kinds = sorted({s.get("kind") for s in f["scenes"]})
    print(f"{i}. {f['title']:<32} {words:>3} words · {len(f['script'])} lines · "
          f"{f['voice'].split('-')[-1]:<14} · scenes: {', '.join(kinds)}")
    print(f"   {f['premise']}")

## 5 · 🏭 Run the factory — five films, end to end

Per film: **director → build → TTS → adaptive timing (one gap retry lands the master at exactly
the target) → music → SFX → mix → audio QA → 720p draft → contact sheet → Groq vision QA →
final render on both T4s → mux → check → YouTube copy.** Stages that fail are recorded; the
factory moves on to the next film.

In [ ]:
import logging, time
logging.basicConfig(level=logging.INFO, format="%(levelname)-7s %(name)s: %(message)s")
for noisy in ("factory.pipeline", "factory.director", "factory.groq"):
    logging.getLogger(noisy).setLevel(logging.WARNING if noisy != "factory.groq" else logging.INFO)

from pathlib import Path
from factory.pipeline import Factory, FactoryConfig

cfg = FactoryConfig(
    target_seconds=TARGET_SECONDS,
    final_tier=FINAL_TIER,
    draft_qa=DRAFT_VISION_QA and (groq is not None),
    stage_timeout=50 * 60,
)
factory = Factory(Path(OVH), Path(FACTORY_REPO), Path(OUT_DIR), groq=groq, config=cfg)

if not AI_DIRECTOR:
    factory.groq = None  # blueprints only, zero API calls

reports = []
T0 = time.time()
for i, film in enumerate(selected, 1):
    print(f"\n{'='*70}\n🎬 film {i}/{len(selected)}: {film['title']}\n{'='*70}")
    rep = factory.make_film(film)
    reports.append(rep)
    qa = (rep.qa or {}).get("overall")
    print(f"\n[{'OK ' if rep.ok else 'FAIL'}] {film['title']} · {rep.seconds:.0f}s · "
          f"vision QA: {qa if qa is not None else '—'} · "
          f"final: {rep.files.get('final') or rep.error}")

print(f"\n{'='*70}\n🏭 factory done: {sum(r.ok for r in reports)}/{len(reports)} films in {(time.time()-T0)/60:.1f} min")

## 6 · Results — the report

In [ ]:
import json, subprocess
from pathlib import Path

rows = []
for r in reports:
    rows.append({
        "film": r.title, "ok": r.ok, "seconds": round(r.seconds),
        "vision_qa": (r.qa or {}).get("overall"),
        "verdict": (r.qa or {}).get("verdict"),
        "final": r.files.get("final"),
        "check": r.stages.get("deliver", {}).get("check_rc"),
    })
print(json.dumps(rows, indent=1))

ok = [r for r in reports if r.ok]
print(f"\n✅ {len(ok)}/{len(reports)} films delivered to {OUT_DIR}:")
for r in ok:
    p = Path(r.files["final"])
    if p.exists():
        mb = p.stat().st_size / 1e6
        dur = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                              "-of", "csv=p=0", str(p)], capture_output=True, text=True).stdout.strip()
        print(f"  {p.name:<48} {float(dur):6.1f}s  {mb:6.1f} MB")
# vision QA issues worth a look
for r in reports:
    issues = (r.qa or {}).get("issues") or []
    if issues:
        print(f"\nQA notes — {r.title}:")
        for i in issues[:5]:
            print("  -", i)

## 7 · Package the outputs

The finals, YouTube copies, per-film metadata (titles/description/tags — Groq-written when
available, blueprint-written otherwise), and the full run report — copied into one folder and
zipped for download.

In [ ]:
import json, shutil, subprocess
from pathlib import Path

out = Path(OUT_DIR)
pkg = out / "package"
pkg.mkdir(parents=True, exist_ok=True)

meta_all = {}
for r in reports:
    if not r.ok:
        continue
    for key in ("final", "delivered"):
        src = r.files.get(key)
        if src and Path(src).exists():
            shutil.copy2(src, pkg / Path(src).name)
    m = r.metadata or {}
    meta_all[r.slug] = {
        "title_variants": m.get("titles") or [r.title],
        "description": m.get("description"),
        "tags": m.get("tags"),
        "youtube_chapters": [c for c in (r.qa or {}).get("chapters", [])] or None,
    }
(pkg / "metadata.json").write_text(json.dumps(meta_all, indent=1, ensure_ascii=False))
if Path(factory.work / "report.json").exists():
    shutil.copy2(factory.work / "report.json", pkg / "report.json")

# the covers make good thumbnails
for r in reports:
    if r.ok:
        cov_dir = Path(OVH) / "projects" / r.slug / "out" / "covers"
        if cov_dir.is_dir():
            for c in sorted(cov_dir.glob("*.png"))[:2]:
                shutil.copy2(c, pkg / f"{r.slug}-{c.name}")

zip_path = out / "discipline-films.zip"
subprocess.run(["zip", "-q", "-r", str(zip_path), "."], cwd=pkg)
print(f"📦 package: {zip_path} ({zip_path.stat().st_size/1e6:.1f} MB)")
for f in sorted(pkg.iterdir()):
    print("  ", f.name, f"{f.stat().st_size/1e6:.1f} MB" if f.is_file() else "")

## Notes for the next run

- **Re-runs are cheap**: the boot cell is idempotent and caches persist in `/kaggle/working`
  (Node, Chrome, uv packages) — a warm restart takes seconds.
- **Change the topic**: the five blueprints live in `groq-film-factory/factory/films_part*.py`;
  the engine (scene kinds, palettes, motion) is `factory/engine-core.js` + `engine-scenes.js`.
- **Try a new Groq model**: put its id at the front of `TEXT_MODEL_PREFERENCE` — that's the whole change.
- **Quality ladder**: `FINAL_TIER="beast"` renders the 720p one-tier ship grade (about 2× faster
  wall-clock); `"final"` is the full 1080p master.
- **Quota**: ~30 GPU-hours/week on Kaggle; one full five-film run costs roughly 1–2 GPU-hours.